# Trees



```
# This is formatted as code
```

## Regression Trees

* Implementing the Regression Tree Class from scratch using only `NumPy`.

In [ ]:
import numpy as np

def generate_regression_data(n_samples=1000, n_features=8, noise=0.1, random_state=42):
    """Generate synthetic regression data similar to California housing."""
    np.random.seed(random_state)

    X = np.random.randn(n_samples, n_features)

    # Create target with non-linear relationships
    y = (2.5 * X[:, 0] +
         1.8 * X[:, 1] ** 2 +
         -1.2 * X[:, 2] * X[:, 3] +
         0.5 * np.sin(5 * X[:, 4]) +
         0.8 * X[:, 5] +
         -0.3 * X[:, 6] ** 3 +
         1.5 * X[:, 7])
    # Add noise
    y += noise * np.random.randn(n_samples)
    # Scale to reasonable range
    y = (y - y.min()) / (y.max() - y.min()) * 4 + 1

    return X, y


In [ ]:
import numpy as np

class RegressionTree:
    """A decision tree for regression using numpy."""

    def __init__(self, max_depth=None, min_samples_split=2, min_samples_leaf=1):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.tree_ = None

    def fit(self, X, y):
        """Build the regression tree."""
        X = np.asarray(X)
        y = np.asarray(y)
        self.n_features_ = X.shape[1]
        self.tree_ = self._build_tree(X, y, depth=0)
        return self

    def _build_tree(self, X, y, depth):
        """Recursively build the tree."""
        n_samples, n_features = X.shape

        # Create a leaf node if stopping criteria are met
        if (
            n_samples < self.min_samples_split
            or (self.max_depth is not None and depth >= self.max_depth)
            or np.unique(y).shape[0] == 1
        ):
            return {
                "leaf": True,
                "value": float(y.mean())
            }

        best_feature = None
        best_threshold = None
        best_error = np.inf
        best_left_mask = None
        best_right_mask = None

        # Current node error (for checking if splitting helps)
        current_error = np.var(y) * n_samples

        for feature in range(n_features):
            X_col = X[:, feature]

            # Sort values of the feature
            sorted_idx = np.argsort(X_col)
            X_sorted = X_col[sorted_idx]

            # Candidate thresholds: midpoints between unique consecutive values
            unique_vals = np.unique(X_sorted)
            if unique_vals.shape[0] == 1:
                continue
            thresholds = (unique_vals[:-1] + unique_vals[1:]) / 2.0

            for thr in thresholds:
                left_mask = X_col <= thr
                right_mask = ~left_mask

                n_left = np.sum(left_mask)
                n_right = n_samples - n_left

                if (
                    n_left < self.min_samples_leaf
                    or n_right < self.min_samples_leaf
                ):
                    continue

                y_left = y[left_mask]
                y_right = y[right_mask]

                # Weighted sum of variances (equivalent to MSE within each side)
                error = np.var(y_left) * n_left + np.var(y_right) * n_right

                if error < best_error:
                    best_error = error
                    best_feature = feature
                    best_threshold = thr
                    best_left_mask = left_mask
                    best_right_mask = right_mask

        # If no useful split found, make a leaf
        if best_feature is None or best_error >= current_error:
            return {
                "leaf": True,
                "value": float(y.mean())
            }

        # Create internal node
        left_subtree = self._build_tree(X[best_left_mask], y[best_left_mask], depth + 1)
        right_subtree = self._build_tree(X[best_right_mask], y[best_right_mask], depth + 1)

        return {
            "leaf": False,
            "feature_index": best_feature,
            "threshold": float(best_threshold),
            "value": float(y.mean()),
            "left": left_subtree,
            "right": right_subtree,
        }

    def _predict_one(self, x, node):
        if node["leaf"]:
            return node["value"]
        if x[node["feature_index"]] <= node["threshold"]:
            return self._predict_one(x, node["left"])
        else:
            return self._predict_one(x, node["right"])

    def predict(self, X):
        """Make predictions for X."""
        X = np.asarray(X)
        return np.array([self._predict_one(x, self.tree_) for x in X])


In [ ]:
X, y = generate_regression_data()
print(X.shape, y.shape)

tree = RegressionTree(max_depth=5)
tree.fit(X, y)
print(tree.tree_)

(1000, 8) (1000,)
{'leaf': False, 'feature_index': 0, 'threshold': 0.001837842881461963, 'value': 2.363071314857283, 'left': {'leaf': False, 'feature_index': 1, 'threshold': -1.712574481478605, 'value': 2.163445490919792, 'left': {'leaf': False, 'feature_index': 1, 'threshold': -2.197403263730698, 'value': 2.9056255818388403, 'left': {'leaf': False, 'feature_index': 1, 'threshold': -2.3700115947989167, 'value': 3.2431313431781286, 'left': {'leaf': False, 'feature_index': 2, 'threshold': 0.08329716275701642, 'value': 3.3102933101982353, 'left': {'leaf': True, 'value': 3.331694567980602}, 'right': {'leaf': True, 'value': 3.2032870212864024}}, 'right': {'leaf': False, 'feature_index': 0, 'threshold': -0.9335154547908943, 'value': 3.04164544211781, 'left': {'leaf': True, 'value': 2.944550613131809}, 'right': {'leaf': True, 'value': 3.138740271103811}}}, 'right': {'leaf': False, 'feature_index': 5, 'threshold': -0.35854396836986213, 'value': 2.712765146787819, 'left': {'leaf': False, 'featu



```
# This is formatted as code
```

## Bagging

* Implementing Bagging using only `NumPy`.
* Comparing the results between the bagged run of the `RegressionTree` class on the synthetic dataset.

In [ ]:
class BaggingRegressor:
    """Bagging ensemble for regression trees."""

    def __init__(
        self,
        n_estimators=10,
        max_samples=1.0,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=None,
    ):
        self.n_estimators = n_estimators
        self.max_samples = max_samples
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.random_state = random_state
        self.estimators_ = []

    def fit(self, X, y):
        """Fit the bagging ensemble."""
        X = np.asarray(X)
        y = np.asarray(y)
        n_samples = X.shape[0]

        # Determine bootstrap sample size
        if 0 < self.max_samples <= 1.0:
            n_bootstrap = int(self.max_samples * n_samples)
        else:
            n_bootstrap = int(self.max_samples)
            n_bootstrap = max(1, min(n_bootstrap, n_samples))

        rng = np.random.RandomState(self.random_state)
        self.estimators_ = []

        for _ in range(self.n_estimators):
            indices = rng.randint(0, n_samples, size=n_bootstrap)
            X_sample = X[indices]
            y_sample = y[indices]

            tree = RegressionTree(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                min_samples_leaf=self.min_samples_leaf,
            )
            tree.fit(X_sample, y_sample)
            self.estimators_.append(tree)

        return self

    def predict(self, X):
        """Make predictions by averaging all trees."""
        X = np.asarray(X)
        # Shape: (n_estimators, n_samples)
        all_preds = np.array([est.predict(X) for est in self.estimators_])
        return all_preds.mean(axis=0)


In [ ]:
# Generate data
X, y = generate_regression_data(n_samples=1000, n_features=8, noise=0.1, random_state=42)

# Train / test split using NumPy only
rng = np.random.RandomState(0)
indices = np.arange(X.shape[0])
rng.shuffle(indices)
split = int(0.8 * len(indices))
train_idx, test_idx = indices[:split], indices[split:]

X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]

# Single tree
tree = RegressionTree(max_depth=5)
tree.fit(X_train, y_train)
y_pred_tree = tree.predict(X_test)
mse_tree = np.mean((y_test - y_pred_tree) ** 2)

# Bagging of trees
bagger = BaggingRegressor(
    n_estimators=25,
    max_samples=0.8,
    max_depth=5,
    random_state=0,
)
bagger.fit(X_train, y_train)
y_pred_bag = bagger.predict(X_test)
mse_bag = np.mean((y_test - y_pred_bag) ** 2)

print("MSE single tree:", mse_tree)
print("MSE bagging:    ", mse_bag)


MSE single tree: 0.08948596226952991
MSE bagging:     0.07688563747926346
